# Predicción exploratoria de fatiga

Datos: [Wiehler et al., Zenodo 6795446](https://doi.org/10.5281/zenodo.6795446). Consulte `README.md` y `docs/methodology.md` antes de interpretar resultados.

Se predice el siguiente autorreporte; no se demuestra anticipación de la conciencia de fatiga. Este notebook llama al mismo flujo que la CLI y debe ejecutarse de principio a fin.

In [ ]:
from pathlib import Path
from fatigue_prediction.cli import run
import pandas as pd

DATA_DIR = Path("data")
OUTPUT_DIR = Path("results/notebook")
FOLDS = 5
SEED = 42

## Preparar los datos

Ejecute `python -m fatigue_prediction download` desde una terminal en la raíz del proyecto. Se requieren los cuatro CSV originales con hashes correctos. La descarga opcional `--all` incluye pupila y decisiones, aún sin modelos. El análisis falla explícitamente si faltan archivos. No utiliza las salidas históricas como resultados.

In [ ]:
frame, comparison, audit = run(
    DATA_DIR, OUTPUT_DIR, folds=FOLDS, seed=SEED,
    algorithms=("linear",), bootstrap=2000,
)
# Añadir "rf" a algorithms para evaluar también Random Forest.
audit

## Cobertura y transiciones

Las respuestas duplicadas se conservan en `ambiguous_ratings.csv`. No se saltan sesiones ni se interpolan medidas fisiológicas.

In [ ]:
display(frame.groupby(["session", "session_future"]).size().rename("n").reset_index())
display(pd.Series(audit["availability"], name="observaciones disponibles"))

## Predicción en participantes no vistos

Compare modelos dentro de cada cohorte y algoritmo. Las cohortes tienen muestras distintas. R² puede ser negativo; MAE/RMSE usan la escala del autorreporte. Persistencia y media del entrenamiento son referencias necesarias.

In [ ]:
display(comparison.round(4))

## Transiciones difíciles

Estos son errores fuera de muestra. No se eliminan las observaciones según su error. Examinar disponibilidad, condición y magnitud del cambio como hipótesis exploratorias, sin atribuir causalidad.

In [ ]:
errors = pd.read_csv(OUTPUT_DIR / "multimodal_linear_errors.csv")
display(errors.head(10))
display(pd.read_csv(OUTPUT_DIR / "multimodal_linear_error_groups.csv"))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(errors["fatigue_future"], errors["pred_M0"], alpha=.7)
ax.plot([0, 100], [0, 100], linestyle="--", color="gray")
ax.set(xlabel="Fatiga futura observada", ylabel="Predicción OOF M0", title="Cohorte multimodal: predicciones de prueba")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "oof_M0.png", dpi=160)
plt.show()

## Incertidumbre y asociación

Los intervalos bootstrap remuestrean sujetos y son condicionales a predicciones OOF ya ajustadas; no cubren toda la incertidumbre de entrenamiento. Las asociaciones OLS agrupadas y el modelo basal con intercepto aleatorio se guardan por separado. Comprobar convergencia y advertencias antes de interpretar.

In [ ]:
display(pd.read_csv(OUTPUT_DIR / "multimodal_linear_M2_conditional_bootstrap.csv", index_col=0))
print((OUTPUT_DIR / "mixed_baseline.txt").read_text())

## Interpretación

Evaluar la mejora frente a persistencia y M0 junto con su incertidumbre, número de sujetos y selección por casos completos. Una mejora exploratoria pequeña no valida un biomarcador; ausencia de mejora no descarta una relación biológica. La cronología real de adquisición requiere confirmación con el protocolo. Los resultados nuevos solo existen después de ejecutar con los CSV verificados.